# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We will use the FAIR^2 dataset which provides clinical and molecular records related to second primary colorectal cancer in cancer survivors.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their @id
print("Available record sets:")
record_sets_list = []
for rs in dataset.record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    record_sets_list.append(rs.id)
    # Print fields for each record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Name: {field.name}, @id: {field.id}, type: {field.data_type}")

### Example: Show a preview of records from a main record set

Let's preview the data for one of the available record sets by referencing it using its `@id`.

In [ ]:
# Choose the first record set for preview
if record_sets_list:
    preview_record_set_id = record_sets_list[0]
    print(f"\nPreview records for record set @id: {preview_record_set_id}")
    for i, x in enumerate(dataset.records(record_set=preview_record_set_id)):
        pprint.pprint(x)
        if i >= 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Record sets and field `@id`s above can be referenced here.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets_list:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if not dataframes:
    print("No dataframes created. No records available.")
else:
    # List all dataframes created, show their columns
    print("\nDataFrames created for these record set @id(s):")
    for rsid, df in dataframes.items():
        print(f"- @id: {rsid} (columns: {df.columns.tolist()})")

    # For demonstration, pick the first available dataframe
    preview_df_key = list(dataframes.keys())[0]
    print(f"\nPreview of first DataFrame (record set @id: {preview_df_key}):")
    display(dataframes[preview_df_key].head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing: filter records, normalize numeric fields, and group data.

- Identify a numeric field by its `@id` from the previously displayed record set.
- Filter based on some threshold.
- Normalize values.
- Group by a categorical field.

In [ ]:
# Identify a numeric field (@id) and a suitable grouping field (@id) for analysis

# Let's auto-detect a suitable numeric field and group field in the first dataframe (heuristic search)
df = dataframes[preview_df_key]
numeric_field_id = None
group_field_id = None

# Heuristic: pick first numeric-like column
for col in df.columns:
    # Try to convert column to numeric
    try:
        x = pd.to_numeric(df[col])
        if not x.isnull().all() and x.nunique() > 1:
            numeric_field_id = col
            break
    except Exception:
        continue
# For grouping, pick a different string/categorical field
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == object and df[col].nunique() < len(df) // 2:
        group_field_id = col
        break

print(f"Numeric field selected for analysis: {numeric_field_id}")
print(f"Group field selected for grouping: {group_field_id}")

# Proceed if found
if numeric_field_id is not None:
    # Convert to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].median() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (records: {len(filtered_df)}):")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        # Show groupby mean of numeric field
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (top 10 groups):")
        display(grouped_df.head(10))
else:
    print("Could not identify a suitable numeric field for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and relation to the chosen group field, if any.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If grouped_df is available, barplot group means
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f'Grouped mean {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 colorectal cancer survivor dataset using the `mlcroissant` library. We identified available record sets and fields by their `@id`s, loaded and previewed real-world clinical and molecular data, selected numeric and categorical fields for analysis, performed normalization, grouping, and created simple visualizations. This notebook demonstrates reproducible and schema-driven data science workflows on FAIR-compliant datasets using Croissant.